In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os

drive_path = "/content/drive/MyDrive"

print(os.listdir(drive_path))

['WhatsApp Image 2026-03-13 at 5.03.18 PM.jpeg', 'WhatsApp Image 2026-03-13 at 5.03.17 PM.jpeg', 'WhatsApp Image 2026-03-13 at 5.03.17 PM (1) (1).jpeg', 'WhatsApp Image 2026-03-13 at 5.03.17 PM (2).jpeg', 'WhatsApp Image 2026-03-13 at 5.03.17 PM (1).jpeg', 'Documents.pdf', 'A.pdf', 'Blank.jpeg', 'Ravi_Gupta_Resume.pdf', 'Ravi Gupta I.Tech Club President Resume.pdf', 'IMDB Dataset.csv.zip', 'Colab Notebooks']


In [ ]:

import zipfile
import os

zip_path = "/content/drive/MyDrive/IMDB Dataset.csv.zip"
extract_path = "/content/imdb"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))


Dataset extracted successfully!
['IMDB Dataset.csv']


In [ ]:

import pandas as pd

csv_path = "/content/imdb/IMDB Dataset.csv"

df = pd.read_csv(csv_path)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nSentiment counts:")
print(df["sentiment"].value_counts())


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset shape: (50000, 2)

Sentiment counts:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:

import re
import html

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["clean_review"] = df["review"].apply(clean_text)

print("Original Review:")
print(df["review"].iloc[1])

print("\nCleaned Review:")
print(df["clean_review"].iloc[1])


Original Review:
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terr

In [ ]:

df["sentiment_encoded"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print(df[["sentiment", "sentiment_encoded"]].head(10))


  sentiment  sentiment_encoded
0  positive                  1
1  positive                  1
2  positive                  1
3  negative                  0
4  positive                  1
5  positive                  1
6  positive                  1
7  negative                  0
8  negative                  0
9  positive                  1


In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token="<OOV>")

tokenizer.fit_on_texts(df["clean_review"])

sequences = tokenizer.texts_to_sequences(df["clean_review"])

print("Original Review:")
print(df["clean_review"].iloc[0])

print("\nToken IDs:")
print(sequences[0][:30])

print("\nVocabulary Size:", len(tokenizer.word_index) + 1)


Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me. The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word. It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away. I would say the main appeal of the show is due to the fact that it goes w

In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_review"])


sequences = tokenizer.texts_to_sequences(df["clean_review"])


max_length = 200

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

y = df["sentiment_encoded"].values

print("Input shape:", X.shape)
print("Output shape:", y.shape)
print("First padded review:", X[0][:30])


Input shape: (50000, 200)
Output shape: (50000,)
First padded review: [  27    5    2   79 2103   45 1072   12  100  147   39  307 3184  398
  475   26 3195   33   23  202   14   11    7  623   48  596   16   68
    2   87]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (40000, 200)
Testing data: (10000, 200)
Training labels: (40000,)
Testing labels: (10000,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(200,)),
    Embedding(input_dim=20000, output_dim=128, mask_zero=True),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,609,473 (9.95 MB)

 Trainable params: 2,609,473 (9.95 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.8181 - loss: 0.4059 - val_accuracy: 0.8555 - val_loss: 0.3427
Epoch 2/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9082 - loss: 0.2413 - val_accuracy: 0.8420 - val_loss: 0.3494
Epoch 3/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9403 - loss: 0.1637 - val_accuracy: 0.8668 - val_loss: 0.3556
Epoch 4/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9573 - loss: 0.1206 - val_accuracy: 0.8608 - val_loss: 0.4036
Epoch 5/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9722 - loss: 0.0848 - val_accuracy: 0.8618 - val_loss: 0.5423


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", round(test_accuracy * 100, 2), "%")
print("Test Loss:", round(test_loss, 4))


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8695 - loss: 0.5170
Test Accuracy: 86.95 %
Test Loss: 0.517


In [24]:
import os

save_dir = "/content/drive/MyDrive/Text_Sentiment_LSTM"
os.makedirs(save_dir, exist_ok=True)

model.save(save_dir + "/sentiment_lstm.keras")

print("Model saved successfully!")

Model saved successfully!


In [25]:
import os
import pickle

from google.colab import drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/Text_Sentiment_LSTM"
os.makedirs(save_dir, exist_ok=True)

with open(save_dir + "/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tokenizer saved successfully!
